## 3.3 Synthetic Regression Data

Machine learning is all about extracting information from data. While we might not care intrinsically about the patterns we ourselves baked into an artificial data generating model, synthetic datasets are nevertheless useful for didactic purposes:

- Evaluating the properties of our learning algorithms.
- Confirming that our implementations work as expected.

For example, if we create data for which the correct parameters are known a priori, we can check that our model can in fact recover them.

In [1]:
%matplotlib inline
import random          # used to shuffle minibatch indices in 3.3.2
import torch
from d2l import torch as d2l

### 3.3.1 Generating Synthetic Data

We work in low dimension ($d = 2$) for succinctness, drawing each row of the feature matrix $\mathbf{X}$ from a **standard normal distribution**. Labels then follow a linear generative model with additive Gaussian noise:

$$\mathbf{y} = \mathbf{X}\mathbf{w} + b + \boldsymbol{\epsilon}, \qquad \epsilon_i \sim \mathcal{N}(0, \sigma^2)$$

Because *we* choose $\mathbf{w}$, $b$, and $\sigma$ (the `noise` argument, default $0.01$) ourselves, we know the **ground truth** — a luxury real data never affords. That is what makes synthetic data useful for didactic purposes: later we can check whether a fitting procedure actually recovers the parameters we already know are correct. The class below adds this generation logic to the `__init__` of a `d2l.DataModule` subclass; `save_hyperparameters()` stashes every constructor argument (`w`, `b`, `noise`, `num_train`, `num_val`, `batch_size`) as an attribute of `self`.

In [2]:
class SyntheticRegressionData(d2l.DataModule):  #@save
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32):
        super().__init__()
        self.save_hyperparameters()  # saves w, b, noise, num_train, num_val, batch_size as self attributes
        n = num_train + num_val
        self.X = torch.randn(n, len(w))          # random input features, shape (n, num_features)
        noise = torch.randn(n, 1) * noise         # gaussian noise scaled by noise level
        self.y = torch.matmul(self.X, w.reshape((-1, 1))) + b + noise  # y = Xw + b + noise

We fix the ground-truth parameters to $\mathbf{w} = [2, -3.4]^\top$ and $b = 4.2$, so later we can check whatever a model *estimates* against these known values. Each row of `data.X` is a length-2 feature vector; each row of `data.y` is the matching scalar label.

In [3]:
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)  # noise defaults to 0.01
print(vars(data).keys())                                          # everything save_hyperparameters() stored
print('features:', data.X[0],'\nlabel:', data.y[0])                # a single (x, y) pair

dict_keys(['hparams', 'root', 'num_workers', 'w', 'b', 'noise', 'num_train', 'num_val', 'batch_size', 'X', 'y'])
features: tensor([ 0.3961, -0.2394]) 
label: tensor([5.8217])


### 3.3.2 Reading the Dataset

Training usually means many passes over the data, grabbing one **minibatch** at a time to update the model. `get_dataloader` below implements that by hand, and registers itself onto `SyntheticRegressionData` via `add_to_class` *after* `data` already exists — Python's object model lets `data` benefit from a method added to its class after the fact. Whether we shuffle matters: for **training** we want a fresh random order each pass (it helps optimization); for **validation** a fixed, predictable order is often more convenient for debugging. The method shuffles indices only when `train=True`, then walks them in `batch_size`-wide windows, `yield`ing one `(X, y)` minibatch at a time — which makes `get_dataloader` a generator, not a plain function.

In [4]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))          # use first num_train samples
        random.shuffle(indices)                            # shuffle for random order each epoch
    else:
        indices = list(range(self.num_train, self.num_train + self.num_val))  # use remaining samples for validation
    for i in range(0, len(indices), self.batch_size):
        batch_indices = torch.tensor(indices[i: i + self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices]  # yield one batch of (X, y) at a time


In [5]:
X, y = next(iter(data.train_dataloader()))                # pull the first minibatch from the generator
print('X shape:', X.shape, '\ny shape:', y.shape)           # (batch_size, 2) features, (batch_size, 1) labels

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


A quick check on the claim above: calling `train_dataloader()` twice should hand back the first few examples in a **different** order each time (fresh shuffle per call), while `val_dataloader()` should hand back the **same** order every time.

In [6]:
first_train  = next(iter(data.train_dataloader()))[0][:3]   # first 3 feature rows, call #1
second_train = next(iter(data.train_dataloader()))[0][:3]   # first 3 feature rows, call #2
first_val    = next(iter(data.val_dataloader()))[0][:3]
second_val   = next(iter(data.val_dataloader()))[0][:3]
print('train order changes:', not torch.equal(first_train, second_train))  # True -- reshuffled each call
print('val order is fixed :', torch.equal(first_val, second_val))          # True -- no shuffling

train order changes: True
val order is fixed : True


### 3.3.3 Concise Implementation of the Data Loader

The hand-rolled generator above is good for building intuition, but it doesn't scale: it needs the *entire* dataset resident in memory and touches it through repeated random-access indexing — a real pipeline must also handle sources that don't fit in memory, live on disk, or stream in on the fly. Rather than writing our own iterator, we lean on PyTorch's own data-loading API: wrap `X` and `y` in a `TensorDataset` (which indexes matching rows of both tensors together) and hand that to a `DataLoader`, which takes care of batching and shuffling efficiently for us. `get_tensorloader` is attached to `d2l.DataModule` itself — not just `SyntheticRegressionData` — so any future `DataModule` whose data already sits in tensors can reuse it as-is; `SyntheticRegressionData.get_dataloader` then becomes a two-line wrapper that just picks the training or validation slice.

In [7]:
@d2l.add_to_class(d2l.DataModule)  #@save
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)          # slice each tensor (X, y) to the requested range
    dataset = torch.utils.data.TensorDataset(*tensors)    # wrap into a PyTorch dataset
    return torch.utils.data.DataLoader(dataset, self.batch_size, shuffle=train)  # shuffle only during training

@d2l.add_to_class(SyntheticRegressionData)  #@save
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)  # pick train or val slice
    return self.get_tensorloader((self.X, self.y), train, i)

In [8]:
X, y = next(iter(data.train_dataloader()))                 # same call, now DataLoader-backed
print('X shape:', X.shape, '\ny shape:', y.shape)            # identical shapes to the from-scratch version

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


The framework's `DataLoader` also supports Python's built-in `len()`, giving the number of batches per epoch directly — no need to compute `ceil(num_train / batch_size)` by hand.

In [9]:
len(data.train_dataloader())  # num_train / batch_size, rounded up -- 1000 / 32 -> 32 batches

32

The whole premise of *synthetic* data, per the intro, is that we know the ground truth and can check it. With `noise=0.01` this tiny linear system is almost exactly determined, so a plain least-squares solve on the full `(X, y)` should recover `w` and `b` to well within a hundredth of their true values.

In [10]:
X_aug = torch.cat([data.X, torch.ones(len(data.X), 1)], dim=1)   # append a column of 1s to fold b into w
w_b_hat = torch.linalg.lstsq(X_aug, data.y).solution               # least-squares solve for [w; b]
print('true   w, b:', data.w, data.b)
print('fitted w, b:', w_b_hat[:-1].squeeze(), w_b_hat[-1].item())

true   w, b: tensor([ 2.0000, -3.4000]) 4.2
fitted w, b: tensor([ 1.9999, -3.4000]) 4.199681282043457


### 3.3.4 Summary

- **Synthetic data** lets us bake in a known generative model $\mathbf{y} = \mathbf{X}\mathbf{w} + b + \boldsymbol{\epsilon}$ and then check whether a learning algorithm actually recovers $\mathbf{w}$ and $b$ — a sanity check that is impossible on real data, where the ground truth is unknown.
- `SyntheticRegressionData` is a `d2l.DataModule`: `save_hyperparameters()` stores every constructor argument on `self`, and `X`, `y` are generated once, in `__init__`.
- The **from-scratch** `get_dataloader` shuffles indices for training but not validation, then `yield`s `batch_size`-wide slices — simple and instructive, but it needs the whole dataset in memory and touches it via random-access indexing.
- The **concise** version wraps `(X, y)` in a `TensorDataset` and a `DataLoader` via `get_tensorloader`, attached to `d2l.DataModule` itself so any tensor-backed `DataModule` can reuse it; a `DataLoader` also supports `len()`, the number of batches per epoch.
- Demonstrated live above: training batches reshuffle on every call while validation batches stay fixed, and a least-squares fit on the full dataset recovers $\mathbf{w}, b$ to well within a hundredth of the true values — exactly the sanity check the introduction promised.
- Up next (§3.4): this same synthetic generator drives a from-scratch implementation of linear regression itself.